# 3. LangGraph 개요 & 상태 머신

**Deep Agents와 create_agent 내부에서 동작하는 LangGraph의 원리를 이해하고, 워크플로우 패턴을 활용한 Agent 제어 흐름을 설계합니다.**

- **3계층 구조** — Deep Agents → LangChain → LangGraph, 각 계층의 역할과 연결 원리
- **State Machine** — `TypedDict` 기반 상태 정의와 그래프 구조로 워크플로 제어
- **StateGraph** — 노드(`Node`), 엣지(`Edge`), 조건부 분기(`Conditional Edge`)로 복잡한 흐름 구성
- **워크플로 패턴** — Prompt Chaining, Parallelization, Routing, Orchestrator-Worker, Evaluator-Optimizer



## 0) 환경 설정
> 진행: [▓░░░░░░░] 1/8


In [1]:
# [3-1] : 라이브러리 설치
# [핵심] LangGraph 및 관련 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행

!uv pip install -q langchain langchain-core langchain-openai langgraph langgraph-checkpoint python-dotenv

In [1]:
# [3-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."

print(f"OPENAI_API_KEY: {'OK' if OPENAI_API_KEY else 'MISSING'} (필수)")

OPENAI_API_KEY: OK (필수)


In [2]:
# [3-4] : 공통 LLM 모델 초기화
# [핵심] 노트북 전체에서 재사용할 모델을 한 곳에서 관리
# [주의] temperature=0 → 결정적 출력; 창의적 작업은 0.7 이상

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1", temperature=0)  # 노트북 전체에서 재사용
print(f"모델 초기화 완료: {llm.model_name}")

모델 초기화 완료: gpt-4.1


In [23]:
from langgraph.graph import StateGraph, START, END, MessagesState
from typing import TypedDict
from langchain.messages import HumanMessage, SystemMessage

# class ChatState(TypedDict):
#     question: str
#     answer: str

def chatbot(state: MessagesState) -> dict:
    """LLM을 호출하고 응답을 메시지 리스트에 추가"""
    llm = ChatOpenAI(model="gpt-4.1", temperature=0)

    msg = [SystemMessage("반말로 답변해줘")] + state["messages"]

    response = llm.invoke(msg)  # 전역 llm 사용
    return {"messages": [response]}  # add_messages 리듀서가 자동 누적

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

res = graph.invoke({"messages": [HumanMessage(content="안녕하세요, 2026년 6월 17일 서울 날씨 어때요?")]})
res


{'messages': [HumanMessage(content='안녕하세요, 2026년 6월 17일 서울 날씨 어때요?', additional_kwargs={}, response_metadata={}, id='38cd3ef8-a92f-476a-a7d3-8b44e13d80fb'),
  AIMessage(content='2026년 6월 17일의 정확한 서울 날씨는 아직 알 수 없어. 내가 가진 정보는 2024년 6월까지의 데이터야.  \n보통 6월 중순 서울은 초여름이라서 낮에는 덥고, 습도도 점점 높아져. 평균 기온은 20~28도 정도고, 장마가 시작될 수도 있어서 비가 올 확률도 있어.  \n정확한 예보는 2026년 가까워져야 알 수 있으니까, 그때 다시 확인하는 게 좋아!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 38, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_9bd9df5354', 'id': 'chatcmpl-DrZ5ERwESHP0NlCasKxDaorMZzQUb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ed321-06dd-7c83-a679-70484c17ea63

{'question': '안녕하세요. 오늘 날씨는 어떤가요?',
 'answer': '안녕하세요! 오늘의 날씨 정보를 알려드리기 위해서는 사용자의 위치(도시 또는 지역명)가 필요합니다. 어느 지역의 날씨를 알고 싶으신가요? 지역을 알려주시면 최신 날씨 정보를 안내해드리겠습니다.'}

## 1) LangGraph 핵심 개념 — 프레임워크 개요와 상태 머신

### LangGraph란?

**LangGraph**는 LangChain 생태계의 **저수준 오케스트레이션 프레임워크**로, **상태 기반 그래프 구조**를 통해 에이전트 워크플로를 정의한다.

### 핵심 특징

- **상태 관리** — `TypedDict` 기반 상태 정의 및 리듀서
- **지속성** — 체크포인터를 통한 상태 자동 저장
- **스트리밍** — 실시간 토큰 단위 출력
- **Human-in-the-loop** — 사람의 개입을 위한 중단/재개
- **내구성 실행** — 장애 발생 시 자동 복구

### 상태 머신(State Machine) 개념

**상태 머신**은 시스템이 **유한한 수의 상태** 중 하나에 있으며, **이벤트(입력)** 에 따라 **다른 상태로 전이**하는 수학적 모델이다. LangGraph는 이 개념을 워크플로에 적용한다:

- **상태(State)** → `TypedDict`로 정의한 공유 데이터
- **노드(Node)** → 상태를 읽고 수정하는 Python 함수
- **엣지(Edge)** → 노드 간 전이 규칙 (고정 또는 조건부)
- **START / END** → 그래프의 진입점과 종료점
> 진행: [▓▓░░░░░░] 2/8

## Deep Agents → LangChain → LangGraph: 3계층 구조

`create_agent()`로 만든 Agent가 내부에서 어떻게 동작하는지를 이해하려면, **3계층 프레임워크 구조**를 알아야 한다.

| 계층 | 역할 | 추상화 수준 | 핵심 API |
|------|------|------------|----------|
| **Deep Agents** | 하네스 (도구+메모리+컨텍스트 자동 조립) | 높음 — 선언적 | `create_agent()`, `AgentConfig` |
| **LangChain** | Agent 조립 (create_agent, @tool, Middleware) | 중간 — 조합적 | `ChatOpenAI`, `@tool`, `ToolNode` |
| **LangGraph** | 제어 흐름 (StateGraph, Node, Edge, 상태 관리) | 낮음 — 명시적 | `StateGraph`, `add_node`, `add_edge` |

> **1일차에 Deep Agents로 체험하고 LangChain으로 조립한 Agent가, 내부적으로는 LangGraph의 StateGraph로 동작합니다.**
> 이제 그 내부 원리를 직접 확인합니다.

### create_agent 내부에서 LangGraph가 동작하는 원리

```
create_agent(tools, model, ...)
    └─ LangChain: ChatOpenAI + ToolNode 조립
        └─ LangGraph: StateGraph 생성
            ├─ add_node("agent", call_model)
            ├─ add_node("tools", tool_node)
            ├─ add_conditional_edges("agent", should_continue)
            └─ compile(checkpointer=...)
```

즉, **선언적으로 작성한 Agent 설정**이 내부에서는 **StateGraph의 노드와 엣지**로 변환된다. 이 노트북에서는 이 변환 과정을 직접 구현해본다.


In [5]:
# [3-5] : LangGraph 구성 요소 확인 — 핵심 임포트
# [핵심] StateGraph, START, END는 모든 LangGraph 그래프의 기본 구성 요소
# [주의] langgraph.graph에서 임포트 — langchain이 아님

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class HelloState(TypedDict):
    """그래프 상태 정의 — 노드 간 데이터 전달 구조"""
    name: str  
    greeting: str  

In [6]:
def greet(state: HelloState) -> dict:
    """노드 함수 — 상태를 읽고 부분 업데이트를 반환"""
    return {"greeting": f"안녕하세요, {state['name']}님! LangGraph에 오신 것을 환영합니다."}

In [7]:
# 그래프 빌드 5단계: StateGraph → add_node → add_edge → compile → invoke
builder = StateGraph(HelloState)        # 1. 상태 스키마로 빌더 생성
builder.add_node("greet", greet)        # 2. 노드(함수) 등록
builder.add_edge(START, "greet")        # 3. START → greet 연결
builder.add_edge("greet", END)          # 4. greet → END 연결
graph = builder.compile()               # 5. 실행 가능한 그래프 생성

In [8]:
result = graph.invoke({"name": "SK하이닉스"})  # 그래프 실행
print(result["greeting"])

안녕하세요, SK하이닉스님! LangGraph에 오신 것을 환영합니다.


## 2) StateGraph 기본 구조 — 노드, 엣지, 상태 정의

### StateGraph 빌드 패턴

```
StateGraph(State) → add_node() → add_edge() → compile() → invoke()
```

| 메서드 | 역할 |
|--------|------|
| `StateGraph(State)` | 상태 스키마로 그래프 빌더 생성 |
| `add_node(name, fn)` | 노드(Python 함수) 등록 |
| `add_edge(src, dst)` | 노드 간 고정 연결 |
| `compile()` | 실행 가능한 `CompiledGraph` 생성 |
| `invoke(input)` | 그래프 실행 — 입력 상태를 넣고 최종 상태를 반환 |

### 노드 함수 규칙

- **인자**: `state: State` — 현재 그래프 상태 전체를 받음
- **반환**: `dict` — 업데이트할 필드만 포함 (부분 업데이트)
- 반환하지 않은 필드는 **기존 값 유지**
> 진행: [▓▓▓░░░░░] 3/8


In [9]:
# [3-6] : 멀티노드 그래프 — 순차 파이프라인
# [핵심] 여러 노드를 엣지로 연결하면 순차 파이프라인이 됨 — START → A → B → END
# [주의] 노드 함수는 dict를 반환 — 업데이트할 필드만 포함

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class TextState(TypedDict):
    text: str           # 원본 텍스트
    word_count: int     # 단어 수
    summary: str        # 요약

In [10]:
def count_words(state: TextState) -> dict:
    """단어 수 계산 노드"""
    word_count = len(state["text"].split())
    return {"word_count": word_count}   

In [11]:
def summarize(state: TextState) -> dict:
    """텍스트 요약 노드 — 단어 수도 활용 가능"""
    summary = f"'{state['text'][:30]}...' (총 {state['word_count']} 단어)"
    return {"summary": summary} 

In [12]:
# 그래프 빌드: START → count → summarize → END
builder = StateGraph(TextState)
builder.add_node("count", count_words)
builder.add_node("summarize", summarize)
builder.add_edge(START, "count")
builder.add_edge("count", "summarize")
builder.add_edge("summarize", END)
graph = builder.compile()

In [13]:
result = graph.invoke({"text": "LangGraph는 상태 머신 기반의 그래프 라이브러리입니다. 노드 간에 상태를 전달하며 복잡한 로직을 구현할 수 있습니다."})
print(f"텍스트: {result['text']}")
print(f"단어 수: {result['word_count']}")
print(f"요약: {result['summary']}")

텍스트: LangGraph는 상태 머신 기반의 그래프 라이브러리입니다. 노드 간에 상태를 전달하며 복잡한 로직을 구현할 수 있습니다.
단어 수: 15
요약: 'LangGraph는 상태 머신 기반의 그래프 라이브러리...' (총 15 단어)


In [ ]:
# [3-7] : 입출력 스키마 분리 — 내부 상태를 외부에 노출하지 않기
# [핵심] input_schema / output_schema로 그래프의 입출력을 내부 상태와 분리
# [주의] 출력 스키마에 없는 필드(intermediate)는 invoke 결과에 포함되지 않음

class InternalState(TypedDict):
    text: str
    word_count: int
    summary: str

class OutputState(TypedDict):
    summary: str

In [ ]:
def process(state: InternalState) -> dict:
    """중간 처리 — 내부 상태에만 저장"""
    word_count = len(state["text"].split())
    return {"word_count": word_count}

In [ ]:
def answer(state: InternalState) -> dict:
    """최종 답변 — 중간 결과 활용"""
    summary = f"'{state['text'][:30]}...' (총 {state['word_count']} 단어)"
    return {"summary": summary}
    

In [ ]:
builder = 
builder.add_node("process", process)
builder.add_node("answer", answer)
builder.add_edge(START, "process")
builder.add_edge("process", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

In [ ]:
result = 
print(f"출력: {result}")  # intermediate 필드 없음

출력: {'answer': "'LANGGRAPH 상태 머신'에 대한 분석 완료"}


## 3) 상태 리듀서 — `Annotated` + `operator.add`, `MessagesState`

### 리듀서(Reducer)란?

**리듀서**는 상태 필드가 **어떻게 업데이트되는지** 결정하는 함수다.

| 리듀서 | 동작 | 예시 |
|--------|------|------|
| 없음 (기본) | **덮어쓰기** — 새 값이 기존 값을 대체 | `count: int` |
| `operator.add` | **누적** — 리스트 항목을 append | `items: Annotated[list, operator.add]` |
| `add_messages` | **메시지 누적** — LLM 대화 히스토리 관리 | `messages: Annotated[list, add_messages]` |
| 커스텀 함수 | 자유 정의 | `Annotated[int, my_reducer]` |

> ⚠️ 리스트 필드에 리듀서 없이 값을 반환하면 **전체가 교체**된다. 누적이 필요하면 반드시 `Annotated`를 사용할 것.
> 진행: [▓▓▓▓░░░░] 4/8


In [ ]:
# [3-8] : operator.add 리듀서 — 리스트 누적 vs 덮어쓰기 비교
# [핵심] Annotated[list, operator.add]는 노드가 반환한 리스트를 기존에 append
# [주의] 리듀서 없는 필드(count)는 마지막 노드의 값으로 덮어써짐

from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END

class AccState(TypedDict):
    

In [ ]:
def step_a(state: AccState) -> dict:
    """첫 번째 노드 — items에 "from_a" 추가, count를 1로 설정"""
    return 

In [ ]:
def step_b(state: AccState) -> dict:
    """두 번째 노드 — items에 "from_b" 추가, count를 2로 설정"""
    return 

In [ ]:
builder = 

graph = builder.compile()

In [ ]:
result = 

print(f"items (리듀서=add): {result['items']}")   # ["from_a", "from_b"] — 누적
print(f"count (덮어쓰기):   {result['count']}")    # 2 — 마지막 값

items (리듀서=add): ['from_a', 'from_b']
count (덮어쓰기):   2


In [ ]:
# [3-9] : MessagesState — LLM 대화용 사전 정의 상태
# [핵심] MessagesState는 messages: Annotated[list, add_messages]를 내장 — LLM 대화 히스토리 자동 관리
# [주의] add_messages 리듀서가 메시지 ID 기반으로 중복 방지 및 순서 유지

from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.messages import HumanMessage, SystemMessage

def chatbot(state: MessagesState) -> dict:
    """LLM을 호출하고 응답을 메시지 리스트에 추가"""
    
    return 

In [ ]:
builder = 

graph = builder.compile()

In [ ]:
result = 

print(f"질문: {result['messages'][0].content}")
print(f"응답: {result['messages'][-1].content}")

질문: LangGraph가 무엇인지 한 문장으로 설명해주세요.
응답: LangGraph는 여러 언어 모델과 도구들을 그래프 형태로 연결하여 복잡한 워크플로우를 쉽게 구축하고 실행할 수 있게 해주는 오픈소스 프레임워크입니다.


## 4) 조건부 엣지 & 라우팅 — `add_conditional_edges`

### 조건부 엣지란?

**고정 엣지**(`add_edge`)는 항상 같은 다음 노드로 이동하지만, **조건부 엣지**(`add_conditional_edges`)는 **상태에 따라 다른 노드로 분기**한다.

```
add_conditional_edges(source_node, routing_function)
```

- `routing_function` — 현재 상태를 받아 **다음 노드 이름**(문자열)을 반환
- `Literal` 타입 힌트를 사용하면 시각화에 가능한 경로가 표시됨
- `END`를 반환하면 그래프 종료
> 진행: [▓▓▓▓▓░░░] 5/8


In [29]:
# [3-10] : 조건부 엣지 — 키워드 기반 라우팅
# [핵심] add_conditional_edges로 상태에 따라 다른 노드로 분기 — 상태 머신의 전이 규칙
# [주의] 라우팅 함수의 반환값은 등록된 노드 이름과 정확히 일치해야 함

from typing import Literal
from langgraph.graph import StateGraph, START, END

class RouterState(TypedDict):
    query:str
    category:str
    result:str

In [80]:
def classify(state: RouterState) -> dict:
    """키워드 기반 분류 — 실무에서는 LLM structured output 사용"""
    q = state["query"].lower()
    res =  llm.invoke(f"""
    다음 질문의 카테고리를 분류해줘. 반드시 "weather", "math", "general" 중에 하나로 말해줘.
    질문 : {q}
    """)

    return {"category": res.content}

In [81]:
def handle_weather(state: RouterState) -> dict:
    return {"result": f"[날씨 전문가] 날씨 정보 조회 중: {state['query']}"}

def handle_math(state: RouterState) -> dict:
    return {"result": f"[수학 전문가] 계산 처리 중: {state['query']}"}

def handle_general(state: RouterState) -> dict:
    return {"result": f"[일반 응답] 처리 중: {state['query']}"}

# 라우팅 함수 — Literal 타입 힌트로 가능한 경로 명시
def route(state: RouterState) -> Literal["weather", "math", "general"]:
    """상태의 category 값에 따라 다음 노드 결정"""
    return state["category"]



In [82]:
# 그래프 빌드
builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("weather", handle_weather)
builder.add_node("math", handle_math)
builder.add_node("general", handle_general)

builder.add_edge(START, "classify")              # 항상 classify 먼저
builder.add_conditional_edges("classify", route)  # 분류 결과에 따라 분기
builder.add_edge("weather", END)                  # 각 핸들러 → 종료
builder.add_edge("math", END)
builder.add_edge("general", END)

graph = builder.compile()

In [83]:
# 3가지 입력으로 테스트
for query in ["오늘 서울 날씨 더하기 3은??", "2+2를 계산해줘", "농담 하나 해줘"]:
    result = graph.invoke({"query": query})
    print(f"  {query} → {result['result']}")

  오늘 서울 날씨 더하기 3은?? → [수학 전문가] 계산 처리 중: 오늘 서울 날씨 더하기 3은??
  2+2를 계산해줘 → [수학 전문가] 계산 처리 중: 2+2를 계산해줘
  농담 하나 해줘 → [일반 응답] 처리 중: 농담 하나 해줘


In [131]:
# [3-11] : LLM 기반 조건부 라우팅 — structured output으로 분류
# [핵심] Pydantic 모델 + with_structured_output으로 LLM이 직접 카테고리를 분류
# [주의] structured output은 모델이 지정된 스키마에 맞는 JSON만 반환하도록 강제

from pydantic import BaseModel, Field
from typing import Literal

class Classification(BaseModel):
    """LLM이 반환할 분류 결과 스키마"""
    category: Literal["technical", "business", "casual"] = Field(
        description="질문의 카테고리"
    )
    

In [132]:
class LLMRouteState(TypedDict):
    question: str
    category: str
    answer: str

In [133]:
def llm_classify(state: LLMRouteState) -> dict:
    """LLM structured output으로 질문 분류"""
    structured_llm = llm.with_structured_output(Classification)  # 스키마 바인딩
    result = structured_llm.invoke(f"다음 질문을 분류하세요: {state['question']}")
    return {"category": result.category}

In [134]:
structured_llm = llm.with_structured_output(Classification)  # 스키마 바인딩
result = structured_llm.invoke(f"다음 질문을 분류하세요: 반도체 식각 공정에 대해 알려줘")
result.category

'technical'

In [124]:
structured_llm = llm.with_structured_output(Classification)  # 스키마 바인딩
# result = structured_llm.invoke(f"다음 질문을 분류하세요: {state['question']}")
res = structured_llm.invoke("EUV 질문에 대해 답변해줘")

In [125]:
def handle_technical(state: LLMRouteState) -> dict:
    r = llm.invoke(f"기술 전문가로서 간결하게 답변: {state['question']}")
    return {"answer": r.content}

def handle_business(state: LLMRouteState) -> dict:
    r = llm.invoke(f"비즈니스 어드바이저로서 간결하게 답변: {state['question']}")
    return {"answer": r.content}

def handle_casual(state: LLMRouteState) -> dict:
    r = llm.invoke(f"친근하게 간결히 답변: {state['question']}")
    return {"answer": r.content}

In [ ]:
def llm_route(state: LLMRouteState):
    return state["category"]

In [140]:
builder = StateGraph(LLMRouteState)
builder.add_node("classify", llm_classify)
builder.add_node("technical", handle_technical)
builder.add_node("business", handle_business)
builder.add_node("casual", handle_casual)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", llm_route)
builder.add_edge("technical", END)
builder.add_edge("business", END)
builder.add_edge("casual", END)

graph = builder.compile()

print(graph.get_graph())

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'classify': Node(id='classify', name='classify', data=classify(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'technical': Node(id='technical', name='technical', data=technical(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'business': Node(id='business', name='business', data=business(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'casual': Node(id='casual', name='casual', data=casual(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='classify', data=None, conditional=False), Edge(source='classify', target='business', data=None, conditional=True), Edge(source='classify', target='casual', 

In [128]:
result = graph.invoke({"question": "반도체 공정에서 EUV 리소그래피의 장점은?"})
print(f"카테고리: {result['category']}")
print(f"답변: {result['answer'][:200]}")

카테고리: technical
답변: EUV 리소그래피의 장점은 다음과 같습니다:

1. **미세 패턴 구현**: 13.5nm 파장의 빛을 사용해 더 작은 회로 패턴을 구현할 수 있습니다.
2. **공정 단순화**: 멀티 패터닝 공정이 줄어들어 생산 효율이 높아집니다.
3. **고집적화**: 트랜지스터 집적도를 높여 칩 성능과 에너지 효율이 향상됩니다.


## 5) 기본 그래프 구성 실습 — compile, invoke, 시각화

### 그래프 실행 방법

| 메서드 | 용도 |
|--------|------|
| `graph.invoke(input)` | 동기 실행 — 최종 상태 반환 |
| `graph.stream(input)` | 스트리밍 — 노드별 중간 결과 순차 반환 |
| `graph.get_graph().draw_mermaid()` | Mermaid 형식 시각화 코드 생성 |
| `graph.get_graph().draw_mermaid_png()` | PNG 이미지 생성 |

### 스트리밍 모드

`stream()`은 각 **슈퍼스텝**(노드 실행) 완료 시 `{노드이름: 상태변경}` 딕셔너리를 yield한다. 이를 통해 그래프 실행 과정을 실시간으로 관찰할 수 있다.
> 진행: [▓▓▓▓▓▓░░] 6/8


In [142]:
# [3-12] : 그래프 시각화 — Mermaid 다이어그램
# [핵심] get_graph().draw_mermaid()로 그래프 구조를 텍스트 다이어그램으로 확인
# [주의] draw_mermaid_png()는 추가 의존성(graphviz 등)이 필요할 수 있음

from langgraph.graph import StateGraph, START, END

class DemoState(TypedDict):
    input: str
    step1_result: str
    step2_result: str
    output: str

def step1(state: DemoState) -> dict:
    return {"step1_result": f"1단계 처리: {state['input']}"}

def step2(state: DemoState) -> dict:
    return {"step2_result": f"2단계 처리: {state['step1_result']}"}

def finalize(state: DemoState) -> dict:
    return {"output": f"최종 결과: {state['step2_result']}"}

builder = StateGraph(DemoState)
builder.add_node("step1", step1)
builder.add_node("step2", step2)
builder.add_node("finalize", finalize)
builder.add_edge(START, "step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile()

# Mermaid 다이어그램 출력
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	step1(step1)
	step2(step2)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> step1;
	step1 --> step2;
	step2 --> finalize;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [144]:
# [3-13] : stream()으로 실행 과정 관찰
# [핵심] stream()은 각 노드 완료 시 {노드이름: 상태변경}을 yield — 실시간 추적 가능
# [주의] invoke()와 달리 최종 상태가 아닌 노드별 변경분이 반환됨

print("=== stream() 실행 과정 ===\n")

for step in graph.stream({"input": "LangGraph 실습"}):
    node_name = list(step.keys())[0]         # 실행된 노드 이름
    updates = step[node_name]                 # 해당 노드의 상태 변경분
    print(f"[{node_name}] → {updates}")

print("\n=== invoke() 최종 결과 ===\n")
final = graph.invoke({"input": "LangGraph 실습"})
print(f"output: {final['output']}")

=== stream() 실행 과정 ===

[step1] → {'step1_result': '1단계 처리: LangGraph 실습'}
[step2] → {'step2_result': '2단계 처리: 1단계 처리: LangGraph 실습'}
[finalize] → {'output': '최종 결과: 2단계 처리: 1단계 처리: LangGraph 실습'}

=== invoke() 최종 결과 ===

output: 최종 결과: 2단계 처리: 1단계 처리: LangGraph 실습


## 6) 워크플로우 패턴 — 5대 핵심 패턴

LangGraph로 구현할 수 있는 **5대 핵심 워크플로 패턴**을 실습한다.

| 패턴 | 구조 | 적합 상황 |
|------|------|----------|
| **Prompt Chaining** | A → B → C (순차) | 단계별 변환 — 번역 → 검증 → 교정 |
| **Parallelization** | A → [B, C] → D (병렬 후 합산) | 독립 분석 — 어조 + 복잡도 동시 분석 |
| **Routing** | A → [B or C or D] (조건 분기) | 분류 기반 처리 — 카테고리별 전문가 |
| **Orchestrator-Worker** | Plan → Send() → Workers (동적) | 동적 하위 작업 — 섹션별 병렬 작성 |
| **Evaluator-Optimizer** | Generate ↔ Evaluate (반복) | 품질 개선 루프 — 슬로건 반복 개선 |
> 진행: [▓▓▓▓▓▓▓░] 7/8

In [ ]:
# [3-14] : Prompt Chaining — 순차적 LLM 호출 파이프라인
# [핵심] 각 단계의 출력이 다음 단계의 입력 — draft → improve → format 순서로 품질 개선
# [주의] 체인이 길수록 지연(latency)과 비용 증가 — 필요한 단계만 포함

from langgraph.graph import StateGraph, START, END

class ChainState(TypedDict):
    

In [ ]:
def generate_draft(state: ChainState) -> dict:
    """1단계: 주제에 대한 초안 생성"""
    response = 
    return 

In [ ]:
def improve_draft(state: ChainState) -> dict:
    """2단계: 초안을 더 매력적으로 개선"""
    response = 
    return 

In [ ]:
builder = 






chain_graph = builder.compile()

In [ ]:
result = 
print(f"초안:   {result['draft']}")
print(f"개선본: {result['improved']}")

초안:   HBM(High Bandwidth Memory)은 기존 DRAM 대비 데이터 처리 속도와 대역폭이 크게 향상된 3차원 적층형 반도체 메모리 기술로, AI, 고성능 컴퓨팅, 그래픽카드 등에서 핵심적으로 사용됩니다.
개선본: HBM(High Bandwidth Memory)은 기존 DRAM보다 최대 수십 배 빠른 데이터 처리 속도와 월등히 넓은 대역폭을 제공하는 3차원 적층형 반도체 메모리 기술로, AI 모델 학습, 슈퍼컴퓨터, 최신 그래픽카드 등 초고성능 시스템의 핵심 부품으로 각광받고 있습니다.


In [185]:
# [3-15] : Parallelization — 독립 태스크의 동시 실행
# [핵심] START에서 여러 노드로 엣지를 연결하면 병렬 실행 — 리듀서로 결과 합산
# [주의] 병렬 노드들은 서로의 결과를 참조할 수 없음 — 독립적인 작업만 병렬화

import operator
from langgraph.graph import StateGraph, START, END

class ParallelState(TypedDict):
    text: str
    analysis: Annotated[list[str], operator.add]  # 병렬 결과 누적

In [186]:
def analyze_tone(state: ParallelState) -> dict:
    """병렬 분석 1: 어조 분석"""
    r = llm.invoke(f"한 문장으로 다음 텍스트의 어조를 분석해주세요: {state['text']}")
    return {"analysis": [f"어조: {r.content}"]}

In [187]:
def analyze_complexity(state: ParallelState) -> dict:
    """병렬 분석 2: 복잡도 평가"""
    r = llm.invoke(f"한 문장으로 다음 텍스트의 복잡도를 평가해주세요: {state['text']}")
    return {"analysis": [f"복잡도: {r.content}"]}


In [188]:
def synthesize(state: ParallelState) -> dict:
    """합산 노드: 병렬 결과를 종합"""
    return {"analysis": [f"종합: {len(state['analysis'])}개 분석 완료"]}

In [189]:
builder = StateGraph(ParallelState)
builder.add_node("tone", analyze_tone)
builder.add_node("complexity", analyze_complexity)
builder.add_node("synthesize", synthesize)

# START에서 두 분석 노드로 병렬 분기
builder.add_edge(START, "tone")
builder.add_edge(START, "complexity")

# 두 노드 완료 후 합성 (LangGraph가 자동으로 두 노드 완료를 대기)
builder.add_edge("tone", "synthesize")
builder.add_edge("complexity", "synthesize")
builder.add_edge("synthesize", END)

parallel_graph = builder.compile()

In [190]:
# 시각화로 병렬 구조 확인
print(parallel_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	tone(tone)
	complexity(complexity)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> complexity;
	__start__ --> tone;
	complexity --> synthesize;
	tone --> synthesize;
	synthesize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [191]:
result = parallel_graph.invoke({"text": "HBM은 고대역폭 메모리로 AI 가속기에 필수적인 핵심 부품입니다.", "analysis": []})
for a in result["analysis"]:
    print(f"  {a}")

  복잡도: 이 문장은 전문 용어(HBM, 고대역폭 메모리, AI 가속기 등)를 포함하고 있으나 문장 구조는 단순하여 중간 정도의 복잡도를 가지고 있습니다.
  어조: 이 문장은 객관적이고 전문적인 어조로 정보를 전달하고 있습니다.
  종합: 2개 분석 완료


In [192]:
# [3-16] : Evaluator-Optimizer — 생성-평가 반복 루프
# [핵심] generate → evaluate → 품질 미달 시 다시 generate — 자동 품질 개선 루프
# [주의] 무한 루프 방지를 위해 반드시 최대 반복 횟수(iterations) 제한 필요

from langgraph.graph import StateGraph, START, END

class EvalState(TypedDict):
    task:str
    draft:str
    feedback:str
    is_good: bool
    iterations : int

In [193]:
def generate(state: EvalState) -> dict:
    """초안 생성 또는 피드백 반영 개선"""
    if state.get("feedback"):
        prompt = f"피드백을 반영하여 개선해주세요.\n원본: {state['draft']}\n피드백: {state['feedback']}"
    else:
        prompt = f"다음에 대한 한 문장 슬로건을 작성해주세요: {state['task']}"
    r = llm.invoke(prompt)
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

In [194]:
def evaluate(state: EvalState) -> dict:
    """초안 평가 — 8/10 이상이면 통과"""
    r = llm.invoke(f"이 슬로건을 1-10으로 평가하고 간단한 피드백을 주세요: '{state['draft']}'")
    content = r.content
    is_good = any(f"{n}/10" in content for n in range(8, 11))  # 8, 9, 10점이면 통과
    return {"feedback": content, "is_good": is_good}

In [195]:
def should_retry(state: EvalState) -> Literal["generate", "__end__"]:
    """조건부 분기 — 품질 통과 또는 최대 반복이면 종료"""
    if state["is_good"] or state["iterations"] >= 3:
        return END
    return "generate"                               # 다시 생성으로 루프

In [196]:
builder = StateGraph(EvalState)
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)
builder.add_edge(START, "generate")                  # 시작 → 생성
builder.add_edge("generate", "evaluate")             # 생성 → 평가
builder.add_conditional_edges("evaluate", should_retry, ["generate", END])  # 평가 → 분기

optimizer = builder.compile()

In [197]:
# Mermaid로 루프 구조 확인
print(optimizer.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	evaluate(evaluate)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	evaluate -.-> __end__;
	evaluate -.-> generate;
	generate --> evaluate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [203]:
result = optimizer.invoke({"task": "AI 반도체 혁신 플랫폼"})
print(f"최종 초안 ({result['iterations']}회 반복): {result['draft']}")

최종 초안 (2회 반복): 피드백을 반영하여 슬로건을 다음과 같이 개선해보았습니다.

---

**개선 슬로건 제안 1:**  
AI 반도체, 미래 혁신의 엔진이 되다

**개선 슬로건 제안 2:**  
세상을 바꾸는 두뇌, AI 반도체 혁신 플랫폼

**개선 슬로건 제안 3:**  
AI 반도체로 여는 미래, 혁신의 플랫폼

**개선 슬로건 제안 4:**  
AI 반도체, 미래를 설계하는 혁신 플랫폼

**개선 슬로건 제안 5:**  
미래를 움직이는 AI 반도체 혁신 플랫폼

---

**개선 방향 설명:**  
- '두뇌' 대신 '엔진', '설계', '움직이는' 등 AI 반도체의 역할을 더 구체적으로 표현했습니다.  
- '미래', '혁신', '플랫폼' 등 기존의 강점을 유지하면서도, 차별화된 이미지를 줄 수 있도록 단어를 조합했습니다.  
- AI 반도체가 미래를 실질적으로 이끌고 변화시키는 주체임을 강조했습니다.

원하시는 분위기나 강조하고 싶은 포인트가 있다면 추가로 맞춤형 슬로건도 제안해드릴 수 있습니다!


In [ ]:
for step in optimizer.stream({'task':"AI를 위한 SK 하이닉스 슬로건"}):
    node_name= list(step.keys())[0]
    updates = step[node_name]
    print(f"[{node_name} -> {updates}]")


[generate -> {'draft': 'AI 혁신의 미래, SK하이닉스와 함께.', 'iterations': 1}]
[evaluate -> {'feedback': '슬로건 평가 (1-10): **7점**\n\n피드백:  \n슬로건이 간결하고, SK하이닉스가 AI 혁신의 미래를 함께 만들어간다는 긍정적인 메시지를 잘 전달하고 있습니다. 다만, 다소 평범하고 차별화된 느낌이 약할 수 있습니다. \'혁신\'과 \'미래\'라는 단어가 자주 사용되는 만큼, SK하이닉스만의 강점이나 구체적인 비전을 조금 더 강조하면 더 인상적일 수 있습니다. 예를 들어, "AI 혁신의 미래, SK하이닉스가 이끕니다"처럼 주도적인 이미지를 줄 수도 있습니다.', 'is_good': False}]
[generate -> {'draft': '개선 슬로건 제안:\n\n**AI 혁신의 미래, SK하이닉스가 이끕니다.**\n\n또는\n\n**AI 혁신을 현실로, SK하이닉스가 앞장섭니다.**\n\n또는\n\n**AI의 내일, SK하이닉스가 만듭니다.**\n\n개선 포인트:\n\n- SK하이닉스의 주도적·선도적 이미지를 강조했습니다.\n- ‘혁신’과 ‘미래’라는 추상적 단어 대신 ‘현실’, ‘내일’ 등으로 구체성을 더했습니다.\n- ‘함께’에서 ‘이끕니다’, ‘앞장섭니다’, ‘만듭니다’ 등 능동적 표현으로 차별화와 임팩트를 강화했습니다.\n\n원하시는 방향이나 추가 피드백이 있다면 말씀해 주세요!', 'iterations': 2}]
[evaluate -> {'feedback': '평가 (1-10): **8점**\n\n**피드백:**\n\n- **장점:**  \n  - SK하이닉스의 주도적이고 선도적인 이미지를 잘 강조했습니다.\n  - ‘이끕니다’, ‘앞장섭니다’, ‘만듭니다’ 등 능동적이고 자신감 있는 표현이 임팩트 있게 다가옵니다.\n  - ‘혁신’과 ‘미래’라는 추상적 단어 대신 ‘현실’, ‘내일’ 등으로 구체성을 더한 점이 긍정적입니다.\n\n- **개선점:**  \n  - 세 슬로건 모

### 5대 워크플로우 패턴 종합

위에서 Prompt Chaining, Parallelization, Evaluator-Optimizer를 실습했다. 여기에 **Orchestrator-Worker**와 **Routing** 패턴을 추가하면 5대 핵심 패턴이 완성된다.

| 패턴 | 흐름 구조 | 결정적 | 병렬 | 반복 | 적합 상황 |
|------|----------|--------|------|------|----------|
| **Prompt Chaining** | A → B → C (순차) | O | X | 순차 | 단계별 변환 (번역→검증→교정) |
| **Parallelization** | A → [B, C] → D (병렬) | O | O | X | 독립 분석 (어조+복잡도 동시) |
| **Routing** | A → [B or C or D] (분기) | O | X | X | 분류 기반 처리 (카테고리별 전문가) |
| **Orchestrator-Worker** | Plan → Send() → Workers (동적) | O | O | X | 동적 하위 작업 (섹션별 병렬 작성) |
| **Evaluator-Optimizer** | Generate ↔ Evaluate (루프) | X | X | O | 품질 개선 루프 (슬로건 반복 개선) |

> **하네스(Deep Agents)의 워크플로 계층**: `create_agent()`가 내부적으로 이 패턴들을 조합하여 Agent의 실행 흐름을 구성한다.
> 예를 들어, Tool 호출 에이전트는 **Routing** + **Evaluator-Optimizer** 패턴의 조합이다.


### Routing 패턴 — LLM 기반 질문 분류와 전문가 분기

**Routing**은 입력을 LLM으로 분류한 뒤, 카테고리에 따라 **서로 다른 전문가 노드**로 분기하는 패턴이다.
`add_conditional_edges`를 사용하여 구현한다.

```
[질문 입력] → classify → ┬─ "technical" → 기술 전문가 → 답변
                         ├─ "business"  → 비즈니스 전문가 → 답변
                         └─ "casual"    → 일상 대화 → 답변
```


In [ ]:
# [3-16a] : Routing 패턴 — LLM 기반 질문 분류와 전문가 분기
# [핵심] classify 노드가 Structured Output으로 카테고리를 분류하고, 조건부 엣지로 전문가 분기
# [주의] with_structured_output으로 반환 타입을 강제 — 라우팅 안정성 확보

from pydantic import BaseModel
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

# --- 분류 스키마 ---
class RouteClassification(BaseModel):
    

In [7]:
# --- 상태 정의 ---
class RouteState(TypedDict):
    question: str
    category: str
    answer: str

In [ ]:
# --- 노드 함수 ---
def classify_question(state: RouteState) -> dict:
    structured = 
    result = 
    return 

def handle_technical(state: RouteState) -> dict:
    r = llm.invoke(f"기술 전문가로서 간결하게 답변해주세요: {state['question']}")
    return {"answer": r.content}

def handle_business(state: RouteState) -> dict:
    r = llm.invoke(f"비즈니스 관점에서 간결하게 답변해주세요: {state['question']}")
    return {"answer": r.content}

def handle_casual(state: RouteState) -> dict:
    r = llm.invoke(f"친근하게 답변해주세요: {state['question']}")
    return {"answer": r.content}

def route_by_category(state: RouteState) -> str:
    return state["category"]

In [ ]:
# --- 그래프 구성 ---
builder = StateGraph(RouteState)
builder.add_node("classify", classify_question)
builder.add_node("technical", handle_technical)
builder.add_node("business", handle_business)
builder.add_node("casual", handle_casual)







router = builder.compile()

In [ ]:
# --- 실행 ---
test_questions = [
    "Python의 GIL은 어떻게 작동하나요?",
    "AI 스타트업 투자 전략은?",
    "오늘 날씨 좋다!",
]
for q in test_questions:
    result = router.invoke({"question": q})
    print(f"[{result['category']:>10s}] {q}")
    print(f"  → {result['answer'][:100]}...")
    print()


[ technical] Python의 GIL은 어떻게 작동하나요?
  → Python의 GIL(Global Interpreter Lock)은 한 번에 하나의 스레드만 파이썬 바이트코드를 실행할 수 있도록 인터프리터 전체에 락을 거는 메커니즘입니다. 이는...

[  business] AI 스타트업 투자 전략은?
  → AI 스타트업 투자 전략은 다음과 같습니다:

1. **핵심 기술력 검증**: 독자적 알고리즘, 데이터, 특허 등 기술적 차별성 확인  
2. **시장 적용성**: 실제 산업 문제...

[    casual] 오늘 날씨 좋다!
  → 맞아요! 오늘 진짜 날씨 너무 좋아요 ☀️ 이런 날엔 산책이나 카페 가기 딱이죠! 혹시 오늘 뭐 할 계획 있으세요? 😄...



### Orchestrator-Worker 패턴 — Send()로 동적 워커 생성

**Orchestrator-Worker**는 오케스트레이터가 작업을 계획하고, `Send()`로 **동적으로 워커를 생성**하여 병렬 처리하는 패턴이다. 작업 수가 런타임에 결정되므로 유연성이 높다.

```
[주제] → plan(섹션 계획) → Send("worker", section) × N → 결과 합산
```


In [239]:
# [3-16b] : Orchestrator-Worker 패턴 — Send()로 동적 작업 분배
# [핵심] plan 노드가 섹션을 결정하고 Send()로 worker에 동적 분배 — 병렬 실행
# [주의] Send()의 두 번째 인자는 worker의 State 스키마와 일치해야 함

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from typing import Annotated, TypedDict
import operator

class OrchestratorState(TypedDict):
    topic : str
    sections : list[str]
    results : Annotated[list[str],operator.add]
    
    
class WorkerState(TypedDict):
    sections : str

In [243]:
def plan_sections(state: OrchestratorState) -> dict:
    r = llm.invoke(
        f"'{state['topic']}'에 대한 짧은 섹션 제목 n개를 나열해주세요. 한 줄에 하나씩, 번호 없이."
    )
    sections = [s.strip() for s in r.content.strip().split("\n") if s.strip()]
    return {"sections": sections}

def assign_workers(state: OrchestratorState) -> list[Send]:
    # return [Send("worker",)]
    return [Send("worker",{"sections":sec}) for sec in state['sections']]

def worker(state: WorkerState) -> dict:
    res = llm.invoke(f"{state['sections']}에 대해 한 문장으로 설명해줘")
    return {"results": [f"## {state['sections']}\n{res.content}"]}

In [244]:
# --- 그래프 구성 ---
builder = StateGraph(OrchestratorState)
builder.add_node("plan", plan_sections)
builder.add_node("worker", worker)

builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", assign_workers, ["worker"])
builder.add_edge("worker", END)

orchestrator = builder.compile()

In [248]:
result = orchestrator.invoke({"topic": "AI 에이전트", "sections": [], "results": []})
for r in result["results"]:
    print(r)
    print()

print(len(result["results"]))

## AI 에이전트란 무엇인가
AI 에이전트란 주어진 목표를 달성하기 위해 환경을 인식하고, 판단하며, 자율적으로 행동하는 인공지능 시스템입니다.

## AI 에이전트의 주요 특징
AI 에이전트는 주어진 목표를 달성하기 위해 환경을 인식하고, 판단하며, 자율적으로 행동하는 인공지능 시스템이다.

## AI 에이전트의 작동 원리
AI 에이전트는 주어진 환경에서 입력을 받아 이를 분석하고, 목표를 달성하기 위해 적절한 행동을 선택하여 실행하는 시스템입니다.

## AI 에이전트의 활용 분야
AI 에이전트는 고객 상담, 데이터 분석, 자동화된 업무 처리, 개인화 추천 등 다양한 분야에서 인간의 작업을 보조하거나 대체하는 역할을 수행합니다.

## AI 에이전트와 인간의 협업
AI 에이전트와 인간의 협업은 서로의 강점을 결합하여 더 효율적이고 창의적인 문제 해결을 이루는 과정입니다.

## AI 에이전트 개발의 도전과제
AI 에이전트 개발의 도전과제는 복잡한 환경에서의 상황 이해, 적응력, 윤리적 판단, 그리고 신뢰성 있는 의사결정 능력을 동시에 확보하는 데 있다.

## 미래의 AI 에이전트 전망
미래의 AI 에이전트는 인간의 다양한 업무를 자율적으로 수행하며, 개인화된 맞춤 서비스와 창의적 문제 해결 능력을 갖춘 지능형 동반자로 발전할 전망입니다.

## AI 에이전트의 윤리적 이슈
AI 에이전트의 윤리적 이슈는 편향, 개인정보 침해, 책임 소재 불분명 등으로 인해 사회적 신뢰와 공정성을 저해할 수 있다는 점이다.

## AI 에이전트와 자동화
AI 에이전트와 자동화는 인공지능이 사람의 개입 없이 스스로 작업을 수행하거나 반복적인 업무를 자동으로 처리하는 기술입니다.

## AI 에이전트의 진화
AI 에이전트의 진화는 단순한 규칙 기반 시스템에서 시작해, 점차 학습과 추론 능력을 갖추고, 자율적으로 복잡한 문제를 해결하며 인간과 자연스럽게 상호작용할 수 있는 지능형 존재로 발전해온 과정이다.

10


## 7) 멀티스텝 대화 흐름 설계 — 상태 기반 다단계 대화

### 체크포인터와 멀티턴 대화

LangGraph의 **체크포인터**(`Checkpointer`)를 사용하면 각 `invoke()` 호출 사이에 **상태가 자동 저장**된다. 이를 통해 멀티턴 대화를 구현할 수 있다.

- `InMemorySaver` — 메모리 기반 체크포인터 (개발/테스트용)
- `thread_id` — 대화 세션 식별자; 같은 `thread_id`로 호출하면 이전 상태 이어받기
- 다른 `thread_id` → 완전히 독립된 새 대화

### 멀티스텝 대화 구조

```
[사용자 입력] → classify → [route] → 전문가 노드 → 응답
                                         ↑ 이전 대화 히스토리 자동 유지
```
> 진행: [▓▓▓▓▓▓▓▓] 8/8


In [265]:
# [3-17] : 체크포인터로 멀티턴 대화 — InMemorySaver
# [핵심] compile(checkpointer=...)로 상태 자동 저장 — 같은 thread_id면 대화 이어감
# [주의] InMemorySaver는 프로세스 종료 시 데이터 소멸 — 프로덕션에서는 DB 기반 체크포인터 사용

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

# 시스템 프롬프트로 역할 부여
SYSTEM_PROMPT = "당신은 반도체 기술 전문 어시스턴트입니다. 간결하고 정확하게 답변하세요."

def chatbot(state: MessagesState) -> dict:
    """시스템 프롬프트 + 대화 히스토리를 LLM에 전달"""
    messages = [SystemMessage(content = SYSTEM_PROMPT)] + state['messages']
    response = llm.invoke(messages)
    return {"messages":[response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

In [266]:
# 체크포인터 연결 — 상태 자동 저장
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [281]:
# 대화 세션 설정
config = {"configurable": {"thread_id": "session-1"}}

In [279]:
# 첫 번째 턴
result1 = graph.invoke(
    {"messages": [HumanMessage(content="6+6은?")]},
    config=config
)
print(f"[턴 1] 응답: {result1['messages'][-1].content[:150]}")

[턴 1] 응답: 6 + 6 = 12입니다.


In [286]:
# 두 번째 턴 — 이전 대화 히스토리가 자동으로 포함됨
result2 = graph.invoke(
    {"messages": [HumanMessage(content="거기에 10곱하기")]},
    config=config
)
print(f"\n[턴 2] 응답: {result2['messages'][-1].content[:150]}")
print(len(result2['messages']))


[턴 2] 응답: 120,000 × 10 = 1,200,000입니다.
18


In [287]:
# 대화 히스토리 확인
print(f"\n총 메시지 수: {len(result2['messages'])}개")


총 메시지 수: 18개


In [264]:
# [3-18] : 멀티스텝 대화 흐름 — 상태 기반 단계별 정보 수집
# [핵심] 커스텀 상태로 대화 단계(phase)를 관리 — 각 단계에서 다른 질문/처리 수행
# [주의] 조건부 엣지로 phase에 따라 다음 노드를 결정 — 상태 머신의 핵심 패턴

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class ConversationState(TypedDict):
    messages: Annotated[list, operator.add]   # 대화 히스토리 (리듀서)
    phase: str                                 # 현재 대화 단계
    user_name: str                             # 수집한 이름
    user_topic: str                            # 수집한 관심 주제
    summary: str                               # 최종 요약

def greet_node(state: ConversationState) -> dict:
    """1단계: 인사 및 이름 요청"""
    return {
        "messages": ["[봇] 안녕하세요! 반도체 학습 어시스턴트입니다. 이름을 알려주세요."],
        "phase": "ask_topic"
    }

def ask_topic_node(state: ConversationState) -> dict:
    """2단계: 이름 저장 후 관심 주제 질문"""
    last_msg = state["messages"][-1]          # 사용자 입력
    return {
        "messages": [f"[봇] {last_msg}님, 반갑습니다! 어떤 반도체 기술에 관심이 있으신가요?"],
        "user_name": last_msg,
        "phase": "generate_summary"
    }

def summary_node(state: ConversationState) -> dict:
    """3단계: 관심 주제로 맞춤 학습 계획 생성"""
    topic = state["messages"][-1]
    response = llm.invoke(f"{state['user_name']}님을 위한 '{topic}' 학습 로드맵을 3줄로 작성해주세요."    )
    return {
        "messages": [f"[봇] {response.content}"],
        "user_topic": topic,
        "summary": response.content,
        "phase": "done"
    }

def route_phase(state: ConversationState) -> str:
    """현재 phase에 따라 다음 노드 결정"""
    return state["phase"]

In [ ]:
builder = StateGraph(ConversationState)
builder.add_node("greet", greet_node)
builder.add_node("ask_topic", ask_topic_node)
builder.add_node("generate_summary", summary_node)

# 시작 → 인사
builder.add_edge(START, "greet")
# 인사 후 → 종료 (사용자 입력 대기)

# ask_topic, generate_summary도 각각 종료 (턴 단위 실행)




memory = InMemorySaver()

# entry_point를 phase에 따라 동적으로 결정하기 위해 별도 래퍼 그래프 사용
# 여기서는 단계별로 직접 invoke하는 방식으로 시연

In [83]:
# 단계 1: 인사
graph_greet = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "multi-step-1"}}
r1 = graph_greet.invoke({"messages": [], "phase": "greet"}, config=config)
print(r1["messages"][-1])

[봇] 안녕하세요! 반도체 학습 어시스턴트입니다. 이름을 알려주세요.


In [ ]:
# 단계 2: 이름 입력 시뮬레이션
print("[사용자] 김엔지니어")

# ask_topic 노드를 직접 빌드하여 실행
builder2 = StateGraph(ConversationState)
builder2.add_node("ask_topic", ask_topic_node)




graph2 = builder2.compile()
r2 = graph2.invoke({**r1, "messages": r1["messages"] + ["김엔지니어"]})
print(r2["messages"][-1])

[사용자] 김엔지니어
[봇] 김엔지니어님, 반갑습니다! 어떤 반도체 기술에 관심이 있으신가요?


In [ ]:
# 단계 3: 관심 주제 입력 시뮬레이션
print("[사용자] HBM 패키징 기술")

builder3 = StateGraph(ConversationState)
builder3.add_node("generate_summary", summary_node)


graph3 = builder3.compile()
r3 = graph3.invoke({**r2, "messages": r2["messages"] + ["HBM 패키징 기술"]})
print(r3["messages"][-1])
print(f"\n수집 정보 — 이름: {r3['user_name']}, 주제: {r3['user_topic']}")

[사용자] HBM 패키징 기술
[봇] 1. HBM(Higher Bandwidth Memory)의 기본 구조와 동작 원리, TSV(Through Silicon Via) 등 핵심 기술 개념을 학습합니다.  
2. HBM 패키징 공정(적층, 본딩, 인터포저, 테스트 등)과 주요 이슈(열 관리, 신호 무결성, 신뢰성 등)를 심층적으로 이해합니다.  
3. 최신 HBM 패키징 트렌드(2.5D/3D 패키징, CoWoS, Foveros 등)와 실제 적용 사례를 분석하여 실무 역량을 강화합니다.

수집 정보 — 이름: 김엔지니어, 주제: HBM 패키징 기술


In [291]:
# [3-19] : 통합 실습 — 라우팅 + 멀티턴 + 상태 관리
# [핵심] 실전 패턴: 분류 → 전문가 라우팅 → 멀티턴 대화를 하나의 그래프로 통합
# [주의] MessagesState + 커스텀 필드 결합 시 TypedDict 상속으로 확장

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage, AIMessage

class AssistantState(MessagesState):
    """MessagesState 확장 — 라우팅용 카테고리 추가"""
    category: str

def classify_intent(state: AssistantState) -> dict:
    """사용자 의도 분류 — LLM structured output"""
    structured =  llm.with_structured_output(AssistantState)  # 스키마 바인딩
    last_msg = state['messages']
    result = structured.invoke(f"다음 질문을 분류하세요: {last_msg}. 반드시 technical, business, casual 중에서로 선택해주세요")
    return {"category" : result.content}

def technical_expert(state: AssistantState) -> dict:
    """기술 전문가 응답"""
    msgs = state['messages']
    response = llm.invoke(f"당신은 기술 전문가입니다. {msgs}에 대해 답변해주세요")
    return {"messages" : response.content}

def business_advisor(state: AssistantState) -> dict:
    """비즈니스 어드바이저 응답"""
    msgs = state['messages']
    response = llm.invoke(f"당신은 비즈니스 어드바이저입니다. {msgs}에 대해 답변해주세요")
    return {"messages" : response.content}

def casual_chat(state: AssistantState) -> dict:
    """일상 대화 응답"""
    msgs = state['messages']
    response = llm.invoke(f"이것은 일상대화입니다. {msgs}에 대해 답변해주세요")
    return {"messages" : response.content}

def route_to_expert(state: AssistantState) -> Literal["technical", "business", "casual"]:
    return state["category"]

In [ ]:
# 그래프 빌드
builder = StateGraph(AssistantState)
builder.add_node("classify",classify_intent)
builder.add_node("technical",technical_expert)
builder.add_node("business",business_advisor)
builder.add_node("casual",casual_chat)

builder.add_edge(START,"classify")
builder.add_conditional_edges("classify",route_to_expert)
builder.add_edge("technical", END)
builder.add_edge("business", END)
builder.add_edge("casual", END)

graph = builder.compile()

print(graph.get_graph())

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'classify': Node(id='classify', name='classify', data=classify(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'technical': Node(id='technical', name='technical', data=technical(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'business': Node(id='business', name='business', data=business(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'casual': Node(id='casual', name='casual', data=casual(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='classify', data=None, conditional=False), Edge(source='classify', target='business', data=None, conditional=True), Edge(source='classify', target='casual', 

In [ ]:
# 체크포인터 연결 — 상태 자동 저장
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [89]:
# 시각화
print(assistant.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	technical(technical)
	business(business)
	casual(casual)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> business;
	classify -.-> casual;
	classify -.-> technical;
	business --> __end__;
	casual --> __end__;
	technical --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [90]:
# 멀티턴 대화 테스트
config = {"configurable": {"thread_id": "assistant-1"}}

# 턴 1: 기술 질문
r1 = assistant.invoke(
    {"messages": [HumanMessage(content="DRAM과 NAND Flash의 차이점은?")]},
    config=config
)
print(f"[턴 1 - {r1['category']}] {r1['messages'][-1].content[:150]}...")

[턴 1 - technical] DRAM(Dynamic Random Access Memory)과 NAND Flash는 모두 반도체 메모리 소자이지만, 구조, 동작 방식, 용도 등에서 큰 차이가 있습니다. 주요 차이점을 표로 정리하면 다음과 같습니다.

| 구분            | DRAM     ...


In [91]:
# 턴 2: 후속 질문 (이전 맥락 유지)
r2 = assistant.invoke(
    {"messages": [HumanMessage(content="그 중 AI 서버에 더 중요한 것은?")]},
    config=config
)
print(f"\n[턴 2 - {r2['category']}] {r2['messages'][-1].content[:150]}...")

print(f"\n총 대화 메시지: {len(r2['messages'])}개")


[턴 2 - technical] AI 서버에서 **DRAM**과 **NAND Flash**는 모두 필수적인 역할을 하지만, **더 중요한 것은 DRAM**입니다. 그 이유는 다음과 같습니다.

---

## 1. AI 서버에서 DRAM의 역할

- **고속 데이터 처리**: AI 서버는 대용량의 데이...

총 대화 메시지: 4개


---

## 정리

| 항목 | 내용 |
|------|------|
| **3계층 구조** | Deep Agents(선언적) → LangChain(조합적) → LangGraph(명시적), create_agent 내부에서 StateGraph 동작 |
| **다룬 기술** | `StateGraph`, `TypedDict`, `Annotated`, `MessagesState`, `InMemorySaver` |
| **핵심 개념** | 상태 머신(State Machine) 기반 워크플로 — 노드(처리 단위), 엣지(전이 규칙), 상태(공유 데이터) |
| **상태 정의** | `TypedDict`로 스키마 정의, `Annotated` + 리듀서로 업데이트 방식 제어 |
| **조건부 분기** | `add_conditional_edges` + 라우팅 함수로 상태 기반 동적 분기 |
| **5대 워크플로 패턴** | Prompt Chaining(순차), Parallelization(병렬), Routing(분기), Orchestrator-Worker(동적 분배), Evaluator-Optimizer(반복) |
| **멀티턴 대화** | `Checkpointer` + `thread_id`로 대화 상태 자동 저장/복원 |
